In [1]:
# Packages import
import os
current_dir = os.getcwd()
os.chdir('..')
print(f'Moving from {current_dir} to {os.getcwd()}')
import numpy as np
import yaml
import time
from source.utils.masks import *
from source.utils.misc import *
from source.utils.assignment import *
from source.utils.dataset import *
from source.utils.eval import *
from source.utils.testers import *
from source.utils.io import *
from source.core.admm import *

Moving from /work/DNAL/sirera.m/CaP2/sandbox to /work/DNAL/sirera.m/CaP2


In [12]:
# Files to load
config_path = './config/cifar10.yaml' 
partition_path = './config/resnet18-np4.yaml' 

In [13]:
def load_yaml(filepath):
    with open(filepath, 'r') as stream:
        try:
            data = yaml.load(stream, yaml.FullLoader)
        except yaml.YAMLError as exc:
            print(exc)
    return data

In [14]:
# Define config and partition files
configs = load_yaml(config_path)
configs['partition_path'] = partition_path
model = get_model_from_code(configs)

In [15]:
configs = partition_generator(configs, model)

num_partition: {'conv1.weight': 4, 'inputs': 4, 'layer1.0.conv1.weight': 4, 'layer1.0.conv2.weight': 4, 'layer1.1.conv1.weight': 4, 'layer1.1.conv2.weight': 4, 'layer2.0.conv1.weight': 4, 'layer2.0.conv2.weight': 4, 'layer2.0.shortcut.0.weight': 4, 'layer2.1.conv1.weight': 4, 'layer2.1.conv2.weight': 4, 'layer3.0.conv1.weight': 4, 'layer3.0.conv2.weight': 4, 'layer3.0.shortcut.0.weight': 4, 'layer3.1.conv1.weight': 4, 'layer3.1.conv2.weight': 4, 'layer4.0.conv1.weight': 4, 'layer4.0.conv2.weight': 4, 'layer4.0.shortcut.0.weight': 4, 'layer4.1.conv1.weight': 4, 'layer4.1.conv2.weight': 4}
ratio_partition: {'conv1.weight': [1, 1, 1, 1], 'inputs': [1, 1, 1, 1], 'layer1.0.conv1.weight': [1, 1, 1, 1], 'layer1.0.conv2.weight': [1, 1, 1, 1], 'layer1.1.conv1.weight': [1, 1, 1, 1], 'layer1.1.conv2.weight': [1, 1, 1, 1], 'layer2.0.conv1.weight': [1, 1, 1, 1], 'layer2.0.conv2.weight': [1, 1, 1, 1], 'layer2.0.shortcut.0.weight': [1, 1, 1, 1], 'layer2.1.conv1.weight': [1, 1, 1, 1], 'layer2.1.conv2.

In [16]:
configs['comm_costs'] = set_communication_cost(model, configs['partition'],)

In [17]:
#configs['device'] = 'cpu'

In [18]:
input_var = get_input_from_code(configs)

In [19]:
model = model.to(configs['device'])
configs['partition'] = featuremap_summary(model, configs['partition'], input_var)

Inference time per data is 16.350031ms.
conv1.weight 1024
layer1.0.conv1.weight 1024
layer1.0.conv2.weight 1024
layer1.1.conv1.weight 1024
layer1.1.conv2.weight 1024
layer2.0.conv1.weight 256
layer2.0.conv2.weight 256
layer2.0.shortcut.0.weight 256
layer2.1.conv1.weight 256
layer2.1.conv2.weight 256
layer3.0.conv1.weight 64
layer3.0.conv2.weight 64
layer3.0.shortcut.0.weight 64
layer3.1.conv1.weight 64
layer3.1.conv2.weight 64
layer4.0.conv1.weight 16
layer4.0.conv2.weight 16
layer4.0.shortcut.0.weight 16
layer4.1.conv1.weight 16
layer4.1.conv2.weight 16


In [20]:
train_loader, test_loader = get_dataset_from_code(configs['data_code'], configs['batch_size'])

Files already downloaded and verified
Files already downloaded and verified


In [21]:
configs['reassign']

True

## Training

In [22]:
evalHelper = EvalHelper(configs['data_code'])

def test_model(model, criterion, cepoch=0):
        acc = evalHelper.get_accuracy(model, test_loader, criterion, cepoch)
        return acc

In [23]:
def standard_train(configs, cepoch, model, data_loader, criterion, optimizer, scheduler, ADMM=None, masks=None, comm=False):

    batch_acc    = AverageMeter()
    batch_loss   = AverageMeter()
    batch_comm   = AverageMeter()
    evalHelper   = EvalHelper(configs['data_code'])
    
    if comm:
        partition = configs['partition']
    
    if ADMM is not None: 
        admm_initialization(configs, ADMM=ADMM, model=model)
        
    start_time = time.time()
    n_data = configs['batch_size'] * len(data_loader)
    pbar = tqdm(enumerate(data_loader), total=n_data/configs['batch_size'], ncols=150)
    
    if configs['reassign']:
        # 1) Use the built FX graph.
        # 2) Identify all add nodes + their two conv parents -> store in add_pairs.
        partition_dict = configs['partition']
        gm = fx.symbolic_trace(model)

        # 1) node_map: node->"layer_name.weight" for quick lookup
        node_map = build_node_map(gm)
        named_mods = dict(model.named_modules())

        # 2) Identify add pairs
        add_node_pairs_map = build_add_pairs(gm, partition_dict, node_map)
        # This is: add_node -> (layerA, layerB, relationship)

        # 3) Build sets to guide the logic
        conv_in_add = set()
        parents_in_add = set()
        for add_node, (layerA, layerB, rel) in add_node_pairs_map.items():
            conv_in_add.add(layerA)
            conv_in_add.add(layerB)
            if rel == "A_is_parent":
                parents_in_add.add(layerA)
            elif rel == "B_is_parent":
                parents_in_add.add(layerB)
        model_graph = {
            'graph': gm,
            'node_map': node_map,
            'named_mods': named_mods,
            'addition_nodes': add_node_pairs_map,
            'addition_set': conv_in_add,
            'parents': parents_in_add,
        }
    
    for batch_idx, batch in pbar:
           
        data   = ()
        for piece in batch[:-1]:
            data += (piece.float().to(configs['device']),)
        target = batch[-1].to(configs['device'])
        total_loss = 0
        comm_loss = 0
        comp_loss = 0

        data = (torch.cat(data, dim=1),)
        
        optimizer.zero_grad()
        
        if configs['mix_up']:
            data, target_a, target_b, lam = mixup_data(*data, y=target, alpha=configs['alpha'])
        # print('data:', data)
        
        # print(len(data))
        # print(data[0].shape)
        output = model(*data)
        # print('output:', output)
        
        if configs['mix_up']:
            loss = mixup_criterion(criterion, output, target_a, target_b, lam, configs['smooth'])
        else:
            loss = criterion(output, target, smooth=configs['smooth'])
            # loss = criterion(output, target.unsqueeze(1).float())
        # print('xentropy_loss:', loss)
        total_loss += (loss * configs['xentropy_weight'])
        # print('total_loss:', total_loss)
        
        if ADMM is not None:
            z_u_update(configs, ADMM, model, cepoch, batch_idx)  # update Z and U variables
            prev_loss, admm_loss, total_loss = append_admm_loss(ADMM, model, total_loss)  # append admm losses
            
        if comm:
            for (name, W) in model.named_parameters():
                if name in ADMM.prune_ratios:
                    comm_cost = torch.abs(W).to('cpu') * configs['comm_costs'][name].to('cpu')
                    '''
                    v1: abs(W)*comm_cost
                    '''
                    comm_cost = comm_cost.view(comm_cost.size(0), -1).sum()
                    if configs['comm_outsize']:
                        comm_loss += comm_cost*partition[name]['outsize']
                    else:
                        comm_loss += comm_cost
                    
                    '''
                    #v2: further constraint on max(abs(W)*comm_cost)
                    '''
                    '''
                    comm_cost = comm_cost.reshape(W.shape[0], W.shape[1], -1).sum(-1)
                    for i in range(partition[name]['num']):
                        for j in range(partition[name]['num']):
                            if i==j: continue
                            cost_interp = comm_cost[partition[name]['filter_id'][i][:,None],
                                                    partition[name]['channel_id'][j]].sum()
                            #comm_loss += cost_interp*partition[name]['outsize']) #p_{count}
                            comm_loss = max(comm_loss,cost_interp*partition[name]['outsize']) # p_{max}
                            
                            #comp_loss = max(comp_loss, cost_interp*partition[name]['outsize'])
                    '''
                    '''
                    computation cost:
                    for i in range(partition[name]['num']):
                        comp_loss = max(comp_loss, torch.abs(W).view(W.size(0), -1)[partition[name]['filter_id'][i],:].sum())
                    '''
            total_loss += configs['lambda_comm'] * comm_loss + configs['lambda_comp'] * comp_loss
            # print('total_loss:', total_loss)
        
        total_loss.backward() # Back Propagation
        
        # For masked training
        if masks is not None:
            with torch.no_grad():
                for name, W in (model.named_parameters()):
                    if name in masks and W.grad is not None:
                        W.grad *= masks[name]
                        
        optimizer.step()
        
        # adjust learning rate
        if ADMM is not None:
            admm_adjust_learning_rate(optimizer, cepoch, configs)
        else:
            scheduler.step()

        # Reassign neurons to machines
        if configs['reassign'] and (batch_idx+1) % configs['reassign_freq'] == 0:
            #print('Updating assignment')
            update_time = time.time()
            update_assignments(model, configs, model_graph)
            configs['comm_costs'] = set_communication_cost(model, configs['partition'])
            #print(f'Assignment ellapsed {time.time()-update_time} ms')


        acc1 = evalHelper.call(output, target)
        batch_loss.update(loss.item(), target.size(0))
        batch_comm.update(comm_loss.item() if comm_loss else comm_loss, target.size(0))
        batch_acc.update(acc1[0].item(), target.size(0))

        
        # # # preparation log information and print progress # # #
        msg = 'Train Epoch: {cepoch} [ {cidx:5d}/{tolidx:5d} ({perc:2d}%)] Loss:{loss:.4f} CommLoss:{commloss:.4f} Acc:{acc:.4f}'.format(
                        cepoch = cepoch,  
                        cidx = (batch_idx+1)*configs['batch_size'], 
                        tolidx = n_data,
                        perc = int(100. * (batch_idx+1)*configs['batch_size']/n_data), 
                        loss = batch_loss.avg,
                        commloss = batch_comm.avg,
                        acc  = batch_acc.avg,
                    )

        pbar.set_description(msg)
    #print('Training time per epoch is {:.2f}s.'.format(time.time()-start_time))

In [24]:
def standard_train_timing(configs, cepoch, model, data_loader, criterion, optimizer, scheduler, ADMM=None, masks=None, comm=False):

    batch_acc = AverageMeter()
    batch_loss = AverageMeter()
    batch_comm = AverageMeter()
    evalHelper = EvalHelper(configs['data_code'])
    
    if comm:
        partition = configs['partition']
    
    if ADMM is not None: 
        start_time = time.time()
        admm_initialization(configs, ADMM=ADMM, model=model)
        print(f"ADMM initialization time: {time.time() - start_time:.4f}s")
        
    start_time = time.time()
    n_data = configs['batch_size'] * len(data_loader)
    pbar = tqdm(enumerate(data_loader), total=n_data/configs['batch_size'], ncols=150)
    
    if configs['reassign']:
        preprocess_start = time.time()
        partition_dict = configs['partition']
        gm = fx.symbolic_trace(model)

        # 1) node_map: node->"layer_name.weight" for quick lookup
        node_map = build_node_map(gm)
        named_mods = dict(model.named_modules())

        # 2) Identify add pairs
        add_node_pairs_map = build_add_pairs(gm, partition_dict, node_map)
        # This is: add_node -> (layerA, layerB, relationship)

        # 3) Build sets to guide the logic
        conv_in_add = set()
        parents_in_add = set()
        for add_node, (layerA, layerB, rel) in add_node_pairs_map.items():
            conv_in_add.add(layerA)
            conv_in_add.add(layerB)
            if rel == "A_is_parent":
                parents_in_add.add(layerA)
            elif rel == "B_is_parent":
                parents_in_add.add(layerB)
        model_graph = {
            'graph': gm,
            'node_map': node_map,
            'named_mods': named_mods,
            'addition_nodes': add_node_pairs_map,
            'addition_set': conv_in_add,
            'parents': parents_in_add,
        }
        print(f"Graph and model preprocessing time: {time.time() - preprocess_start:.4f}s")
    
    for batch_idx, batch in pbar:
        batch_start_time = time.time()
        total_loss = 0
        comm_loss = 0
        comp_loss = 0
        # Prepare data
        data_prep_start = time.time()
        data = ()
        for piece in batch[:-1]:
            data += (piece.float().to(configs['device']),)
        target = batch[-1].to(configs['device'])
        data = (torch.cat(data, dim=1),)
        print(f"Data preparation time: {time.time() - data_prep_start:.4f}s")
        
        optimizer.zero_grad()
        
        # Forward pass
        forward_start = time.time()
        if configs['mix_up']:
            data, target_a, target_b, lam = mixup_data(*data, y=target, alpha=configs['alpha'])
        output = model(*data)
        print(f"Forward pass time: {time.time() - forward_start:.4f}s")
        
        # Compute loss
        loss_start = time.time()
        if configs['mix_up']:
            loss = mixup_criterion(criterion, output, target_a, target_b, lam, configs['smooth'])
        else:
            loss = criterion(output, target, smooth=configs['smooth'])
        total_loss = loss * configs['xentropy_weight']
        print(f"Loss computation time: {time.time() - loss_start:.4f}s")
        
        if ADMM is not None:
            admm_start = time.time()
            z_u_update(configs, ADMM, model, cepoch, batch_idx)  # update Z and U variables
            prev_loss, admm_loss, total_loss = append_admm_loss(ADMM, model, total_loss)  # append admm losses
            print(f"ADMM update time: {time.time() - admm_start:.4f}s")
            
        if comm:
            comm_start = time.time()
            for (name, W) in model.named_parameters():
                if name in ADMM.prune_ratios:
                    comm_cost = torch.abs(W).to('cpu') * configs['comm_costs'][name].to('cpu')
                    comm_cost = comm_cost.view(comm_cost.size(0), -1).sum()
                    if configs['comm_outsize']:
                        comm_loss += comm_cost * partition[name]['outsize']
                    else:
                        comm_loss += comm_cost
            total_loss += configs['lambda_comm'] * comm_loss + configs['lambda_comp'] * comp_loss
            print(f"Communication loss computation time: {time.time() - comm_start:.4f}s")
        
        backward_start = time.time()
        total_loss.backward()  # Back Propagation
        print(f"Backward pass time: {time.time() - backward_start:.4f}s")
        
        # For masked training
        if masks is not None:
            mask_start = time.time()
            with torch.no_grad():
                for name, W in model.named_parameters():
                    if name in masks and W.grad is not None:
                        W.grad *= masks[name]
            print(f"Mask application time: {time.time() - mask_start:.4f}s")
                        
        optimizer_step_start = time.time()
        optimizer.step()
        print(f"Optimizer step time: {time.time() - optimizer_step_start:.4f}s")
        
        # Adjust learning rate
        scheduler_start = time.time()
        if ADMM is not None:
            admm_adjust_learning_rate(optimizer, cepoch, configs)
        else:
            scheduler.step()
        print(f"Scheduler step time: {time.time() - scheduler_start:.4f}s")

        # Reassign neurons to machines
        if configs['reassign'] and (batch_idx + 1) % configs['reassign_freq'] == 0:
            reassign_start = time.time()
            update_assignments_with_timing(model, configs, model_graph)
            configs['comm_costs'] = set_communication_cost(model, configs['partition'])
            print(f"Reassignment time: {time.time() - reassign_start:.4f}s")
        
        acc1 = evalHelper.call(output, target)
        batch_loss.update(loss.item(), target.size(0))
        batch_comm.update(comm_loss.item() if comm_loss else comm_loss, target.size(0))
        batch_acc.update(acc1[0].item(), target.size(0))

        print(f"Batch {batch_idx + 1} time: {time.time() - batch_start_time:.4f}s")
        print("-"*40)

    print(f"Total training epoch time: {time.time() - start_time:.4f}s")


In [25]:
old = configs['partition']

In [26]:
filepath = os.path.join(os.getcwd(), 'assets', 'models', "cifar10-resnet18.pt")
state_dict = torch.load(filepath, map_location=configs['device'])
model = load_state_dict(model, state_dict['model_state_dict'] if 'model_state_dict' in state_dict 
                               else state_dict['state_dict'] if 'state_dict' in state_dict else state_dict,)

In [27]:
nepoch = configs['epochs']
criterion, optimizer, scheduler = set_optimizer(configs, model, train_loader, \
                                    configs['optimizer'], configs['learning_rate'], nepoch)

# Initializing ADMM; if not admm, do hard pruning only
admm = ADMM(configs, model, rho=configs['rho']) if configs['admm'] else None

best = 0
# prune
for cepoch in range(0, 7):#nepoch+1):
    if cepoch>0:
        print('Learning rate: {:.4f}'.format(get_lr(optimizer)))
        standard_train(configs, cepoch, model, train_loader, 
                    criterion, optimizer, scheduler, ADMM=admm, comm=True)
        if configs['reassign']:
            save_partition(configs, cepoch)
    acc = test_model(model, criterion, cepoch)
    if acc > best and cepoch>0:
        best = acc
        model_name = filepath[:-3]
        model_name = model_name + '-reassign' + str(configs['reassign']) + '2.pt'
        torch.save(model.state_dict(), model_name)
        print('Save model')
    

{'conv1.weight': 0.75, 'layer1.0.conv1.weight': 0.75, 'layer1.0.conv2.weight': 0.75, 'layer1.1.conv1.weight': 0.75, 'layer1.1.conv2.weight': 0.75, 'layer2.0.conv1.weight': 0.75, 'layer2.0.conv2.weight': 0.75, 'layer2.0.shortcut.0.weight': 0.75, 'layer2.1.conv1.weight': 0.75, 'layer2.1.conv2.weight': 0.75, 'layer3.0.conv1.weight': 0.75, 'layer3.0.conv2.weight': 0.75, 'layer3.0.shortcut.0.weight': 0.75, 'layer3.1.conv1.weight': 0.75, 'layer3.1.conv2.weight': 0.75, 'layer4.0.conv1.weight': 0.75, 'layer4.0.conv2.weight': 0.75, 'layer4.0.shortcut.0.weight': 0.75, 'layer4.1.conv1.weight': 0.75, 'layer4.1.conv2.weight': 0.75}
Epoch-[000]: Test loss: 0.24, acc: 93.83.
Learning rate: 0.0100


Train Epoch: 1 [ 50048/50048 (100%)] Loss:0.9079 CommLoss:2559692.6663 Acc:46.7360: 100%|█████████████████████████| 391/391.0 [24:49<00:00,  3.81s/it]


Epoch-[001]: Test loss: 0.39, acc: 87.81.
Save model
Learning rate: 0.0100


Train Epoch: 2 [ 50048/50048 (100%)] Loss:0.8270 CommLoss:1690139.9229 Acc:44.5920: 100%|█████████████████████████| 391/391.0 [11:24<00:00,  1.75s/it]


Epoch-[002]: Test loss: 0.46, acc: 85.61.
Learning rate: 0.0100


...

Epoch-[004]: Test loss: 0.48, acc: 84.80.
Learning rate: 0.0100


Train Epoch: 5 [ 50048/50048 (100%)] Loss:0.8368 CommLoss:903975.9328 Acc:45.3680: 100%|██████████████████████████| 391/391.0 [41:16<00:00,  6.33s/it]


Epoch-[005]: Test loss: 0.46, acc: 85.08.
Learning rate: 0.0100


Train Epoch: 6 [ 50048/50048 (100%)] Loss:0.8702 CommLoss:801200.0886 Acc:46.8300: 100%|██████████████████████████| 391/391.0 [39:02<00:00,  5.99s/it]


Epoch-[006]: Test loss: 0.45, acc: 86.13.


In [28]:
# hard prune
#hard_prune(admm, self.model, self.configs['sparsity_type'], option=None)

# test sparsity
test_kernel_sparsity(model, partition=configs['partition'])
test_partition(model, partition=configs['partition'])

---------------------------------------------------------------------------
total number of zeros: 172947, non-zeros: 10986285, zero sparsity is: 0.0155
total number of kernels:1392832, zero-kernels:19553, kernel sparsity is: 0.0140


conv1.weight:   params:1728,           params-intrap:333,         params-interp:963,           interp-k:144,    interp-k(select):107,   max-interp-k(select):1,     outsize:1024, total-interp-comm:9216, max-interp-comm:1024
[1024, 1024, 1024, 0, 1024, 1024, 0, 1024, 1024, 0, 1024, 1024]
layer1.0.conv1.weight:   params:36864,           params-intrap:6939,         params-interp:20808,           interp-k:3072,    interp-k(select):2312,   max-interp-k(select):16,     outsize:1024, total-interp-comm:180224, max-interp-comm:16384
[14336, 15360, 15360, 13312, 16384, 15360, 14336, 15360, 14336, 14336, 16384, 15360]
layer1.0.conv2.weight:   params:36864,           params-intrap:8928,         params-interp:26784,           interp-k:3072,    interp-k(select):2976,   

In [ ]:
nepoch = configs['epochs']
criterion, optimizer, scheduler = set_optimizer(configs, model, train_loader, \
                                    configs['optimizer'], configs['learning_rate'], nepoch)

# Initializing ADMM; if not admm, do hard pruning only
admm = ADMM(configs, model, rho=configs['rho']) if configs['admm'] else None

best = 0
# prune
for cepoch in range(0, 10):#nepoch+1):
    if cepoch>0:
        print('Learning rate: {:.4f}'.format(get_lr(optimizer)))
        standard_train(configs, cepoch, model, train_loader, 
                    criterion, optimizer, scheduler, ADMM=admm, comm=True)
        if configs['reassign']:
            save_partition(configs, cepoch)
    acc = test_model(model, criterion, cepoch)
    if acc > best and cepoch>0:
        best = acc
        model_name = filepath[:-3]
        model_name = model_name + '-reassign' + str(configs['reassign']) + '4.pt'
        torch.save(model.state_dict(), model_name)
        print('Save model')
    

{'conv1.weight': 0.75, 'layer1.0.conv1.weight': 0.75, 'layer1.0.conv2.weight': 0.75, 'layer1.1.conv1.weight': 0.75, 'layer1.1.conv2.weight': 0.75, 'layer2.0.conv1.weight': 0.75, 'layer2.0.conv2.weight': 0.75, 'layer2.0.shortcut.0.weight': 0.75, 'layer2.1.conv1.weight': 0.75, 'layer2.1.conv2.weight': 0.75, 'layer3.0.conv1.weight': 0.75, 'layer3.0.conv2.weight': 0.75, 'layer3.0.shortcut.0.weight': 0.75, 'layer3.1.conv1.weight': 0.75, 'layer3.1.conv2.weight': 0.75, 'layer4.0.conv1.weight': 0.75, 'layer4.0.conv2.weight': 0.75, 'layer4.0.shortcut.0.weight': 0.75, 'layer4.1.conv1.weight': 0.75, 'layer4.1.conv2.weight': 0.75}
Epoch-[000]: Test loss: 0.24, acc: 93.83.
Learning rate: 0.0100


Train Epoch: 1 [  3200/50048 ( 6%)] Loss:1.6044 CommLoss:3107902.1303 Acc:26.6562:   6%|█▌                       | 25/391.0 [02:38<2:42:33, 26.65s/it]

Finished update_assignments.


Train Epoch: 1 [  6400/50048 (12%)] Loss:1.3773 CommLoss:3090813.5863 Acc:33.3281:  13%|███▏                     | 50/391.0 [05:41<3:00:27, 31.75s/it]

Finished update_assignments.


...

Train Epoch: 6 [ 48000/50048 (95%)] Loss:0.8704 CommLoss:803486.2154 Acc:47.8500:  96%|█████████████████████████▉ | 375/391.0 [33:29<07:12, 27.04s/it]

Finished update_assignments.


Train Epoch: 6 [ 50048/50048 (100%)] Loss:0.8706 CommLoss:802061.5729 Acc:48.8280: 100%|██████████████████████████| 391/391.0 [34:01<00:00,  5.22s/it]


Partition saved to ./config/resnet18-np4_6.yaml
Epoch-[006]: Test loss: 0.46, acc: 85.15.
Learning rate: 0.0100


Train Epoch: 7 [  1920/50048 ( 3%)] Loss:1.0487 CommLoss:738554.9957 Acc:59.4271:   4%|█                           | 15/391.0 [00:30<12:54,  2.06s/it]

In [ ]:
# hard prune
#hard_prune(admm, self.model, self.configs['sparsity_type'], option=None)

# test sparsity
test_kernel_sparsity(model, partition=configs['partition'])
test_partition(model, partition=configs['partition'])

In [ ]:
nepoch = configs['epochs']
criterion, optimizer, scheduler = set_optimizer(configs, model, train_loader, \
                                    configs['optimizer'], configs['learning_rate'], nepoch)

# Initializing ADMM; if not admm, do hard pruning only
admm = ADMM(configs, model, rho=configs['rho']) if configs['admm'] else None

best = 0
# prune
for cepoch in range(0, 7):#nepoch+1):
    if cepoch>0:
        print('Learning rate: {:.4f}'.format(get_lr(optimizer)))
        standard_train(configs, cepoch, model, train_loader, 
                    criterion, optimizer, scheduler, ADMM=admm, comm=True)
        if configs['reassign']:
            save_partition(configs, cepoch)
    acc = test_model(model, criterion, cepoch)
    if acc > best and cepoch>0:
        best = acc
        model_name = filepath[:-3]
        model_name = model_name + '-reassign' + str(configs['reassign']) + '3.pt'
        torch.save(model.state_dict(), model_name)
        print('Save model')
    

{'conv1.weight': 0.75, 'layer1.0.conv1.weight': 0.75, 'layer1.0.conv2.weight': 0.75, 'layer1.1.conv1.weight': 0.75, 'layer1.1.conv2.weight': 0.75, 'layer2.0.conv1.weight': 0.75, 'layer2.0.conv2.weight': 0.75, 'layer2.0.shortcut.0.weight': 0.75, 'layer2.1.conv1.weight': 0.75, 'layer2.1.conv2.weight': 0.75, 'layer3.0.conv1.weight': 0.75, 'layer3.0.conv2.weight': 0.75, 'layer3.0.shortcut.0.weight': 0.75, 'layer3.1.conv1.weight': 0.75, 'layer3.1.conv2.weight': 0.75, 'layer4.0.conv1.weight': 0.75, 'layer4.0.conv2.weight': 0.75, 'layer4.0.shortcut.0.weight': 0.75, 'layer4.1.conv1.weight': 0.75, 'layer4.1.conv2.weight': 0.75}
Epoch-[000]: Test loss: 0.24, acc: 93.83.
Learning rate: 0.0100


Train Epoch: 1 [  1280/50048 ( 2%)] Loss:1.7908 CommLoss:3081169.5028 Acc:27.1875:   3%|▋                          | 10/391.0 [00:22<15:31,  2.45s/it]

Finished update_assignments.


Train Epoch: 1 [  2560/50048 ( 5%)] Loss:1.5885 CommLoss:3090971.6354 Acc:31.3672:   5%|█▍                         | 20/391.0 [00:43<14:13,  2.30s/it]

Finished update_assignments.


...

Finished update_assignments.


Train Epoch: 3 [ 42240/50048 (84%)] Loss:0.8781 CommLoss:1287389.5486 Acc:48.8163:  84%|█████████████████████▉    | 330/391.0 [14:08<02:34,  2.53s/it]

Finished update_assignments.


Train Epoch: 3 [ 43520/50048 (86%)] Loss:0.8816 CommLoss:1284124.2961 Acc:48.7339:  87%|██████████████████████▌   | 340/391.0 [14:31<02:06,  2.48s/it]

Finished update_assignments.


In [ ]:
# hard prune
#hard_prune(admm, self.model, self.configs['sparsity_type'], option=None)

# test sparsity
test_kernel_sparsity(model, partition=configs['partition'])
test_partition(model, partition=configs['partition'])

In [19]:
model_name = filepath[:-3]
model_name = model_name + '-reassign' + str(configs['reassign']) + '2.pt'
torch.save(model.state_dict(), model_name)

In [19]:
new = configs['partition']

In [13]:
from torch.fx import symbolic_trace

def print_fx_graph(model):
    # Symbolically trace the model
    gm = symbolic_trace(model)
    
    print("\n=== FX Graph Nodes ===")
    for node in gm.graph.nodes:
        # node.op: 'placeholder', 'call_module', 'call_function', ...
        # node.target: name of submodule or function
        # node.args: references to other nodes (the inputs)
        # node.kwargs: any keyword args
        print(f"Node: {node.op} - {node.target} - args={node.args}, kwargs={node.kwargs}")

    print("\n=== Module Input Dependencies ===")
    # Let's find 'call_module' nodes and see who their args are
    for node in gm.graph.nodes:
        if node.op == 'call_module':
            print(f"Module '{node.target}' takes input from:")
            for arg in node.args:
                if hasattr(arg, 'target'):
                    print(f"  - {arg.op}:{arg.target}")
                else:
                    print(f"  - {arg}")
    
    # If you specifically want to find who feeds into "layer2.0.shortcut.0.weight"
    # you'd look for a node with target == 'layer2.0.shortcut.0'
    # or a substring if your modules are named differently.

print_fx_graph(model)


=== FX Graph Nodes ===
Node: placeholder - x - args=(), kwargs={}
Node: call_module - conv1 - args=(x,), kwargs={}
Node: call_module - bn1 - args=(conv1,), kwargs={}
Node: call_module - relu - args=(bn1,), kwargs={}
Node: call_module - layer1.0.conv1 - args=(relu,), kwargs={}
Node: call_module - layer1.0.bn1 - args=(layer1_0_conv1,), kwargs={}
Node: call_module - layer1.0.relu - args=(layer1_0_bn1,), kwargs={}
Node: call_module - layer1.0.conv2 - args=(layer1_0_relu,), kwargs={}
Node: call_module - layer1.0.bn2 - args=(layer1_0_conv2,), kwargs={}
Node: call_function - <built-in function add> - args=(layer1_0_bn2, relu), kwargs={}
Node: call_module - layer1.0.relu - args=(add,), kwargs={}
Node: call_module - layer1.1.conv1 - args=(layer1_0_relu_1,), kwargs={}
Node: call_module - layer1.1.bn1 - args=(layer1_1_conv1,), kwargs={}
Node: call_module - layer1.1.relu - args=(layer1_1_bn1,), kwargs={}
Node: call_module - layer1.1.conv2 - args=(layer1_1_relu,), kwargs={}
Node: call_module - lay

## Model testing

In [14]:
train_loader, test_loader = get_dataset_from_code(configs['data_code'], configs['batch_size'])

Files already downloaded and verified
Files already downloaded and verified


In [15]:
filepath = os.path.join(os.getcwd(), 'assets', 'models', "cifar100-resnet101.pt")
state_dict = torch.load(filepath, map_location=configs['device'])
model = load_state_dict(model, state_dict['model_state_dict'] if 'model_state_dict' in state_dict 
                               else state_dict['state_dict'] if 'state_dict' in state_dict else state_dict,)

In [16]:
nepoch=0
model = model.to(configs['device'])
evalHelper = EvalHelper(configs['data_code'])
criterion, optimizer, scheduler = set_optimizer(configs, model, train_loader, \
                                    configs['optimizer'], configs['learning_rate'], nepoch)

In [17]:
acc = evalHelper.get_accuracy(model, test_loader, criterion, 0)

Epoch-[000]: Test loss: 1.26, acc: 70.17.


## Tests

In [7]:
# Partition information
partition = {
    'conv1.weight': {
        'num': 3,  # 3 partitions
        'filter_id': [np.array([0, 1]), np.array([2, 3]), np.array([4, 5])],  # Filters per partition
        'channel_id': [np.array([0, 1]), np.array([2]), np.array([3])],  # Input channels per partition
        'maps': [[0, 1, 2], [1, 0, 1], [2, 1, 0]]  # Communication cost maps
    }
}

# Weights for conv1.weight layer (4D: Conv2D)
weights = torch.tensor([
    [[[1, 0], [0, 0]], [[0, 0], [0, 0]], [[0, 0], [0, 0]], [[0, 0], [0, 0]]],  # Filter 0
    [[[0, 0], [0, 0]], [[1, 0], [0, 0]], [[0, 0], [1, 0]], [[0, 0], [0, 1]]],  # Filter 1
    [[[1, 1], [1, 0]], [[0, 0], [0, 0]], [[1, 0], [0, 1]], [[0, 0], [0, 1]]],  # Filter 2
    [[[0, 0], [1, 0]], [[1, 0], [0, 0]], [[0, 1], [0, 0]], [[0, 0], [0, 0]]],  # Filter 3
    [[[0, 1], [0, 0]], [[0, 0], [1, 1]], [[1, 0], [0, 0]], [[0, 0], [0, 0]]],  # Filter 4
    [[[1, 0], [0, 0]], [[0, 1], [1, 1]], [[0, 0], [0, 0]], [[0, 0], [1, 0]]]   # Filter 5
])



In [9]:
cost_matrix = compute_cost_matrix('conv1.weight', weights, partition)
print(cost_matrix)


[[0. 1. 2.]
 [3. 2. 3.]
 [3. 2. 3.]
 [1. 2. 5.]
 [1. 2. 5.]
 [2. 3. 4.]]


In [15]:
current_time = time.time()
num_layers = 0
print(f'Start time: {current_time} s')
for name, W in model.named_parameters():
    if name in configs['partition']:
        C = compute_cost_matrix(name, W, configs['partition'])
        num_layers += 1
        print(f'Layer {name} processed in: {time.time() - current_time} s')
        print(f'C shape: {len(C)}, {len(C[0])}')
        current_time = time.time()
        
print(f'Num. layers: {num_layers}') 

Start time: 1732648294.427691 s
Layer conv1.weight processed in: 0.020582914352416992 s
C shape: 64, 4
Layer layer1.0.conv1.weight processed in: 0.01493525505065918 s
C shape: 64, 4
Layer layer1.0.conv2.weight processed in: 0.011151790618896484 s
C shape: 64, 4
Layer layer1.1.conv1.weight processed in: 0.009544134140014648 s
C shape: 64, 4
Layer layer1.1.conv2.weight processed in: 0.008366107940673828 s
C shape: 64, 4
Layer layer1.2.conv1.weight processed in: 0.0076978206634521484 s
C shape: 64, 4
Layer layer1.2.conv2.weight processed in: 0.00710606575012207 s
C shape: 64, 4
Layer layer2.0.conv1.weight processed in: 0.01325225830078125 s
C shape: 128, 4
Layer layer2.0.conv2.weight processed in: 0.014045000076293945 s
C shape: 128, 4
Layer layer2.0.shortcut.0.weight processed in: 0.011737823486328125 s
C shape: 128, 4
Layer layer2.1.conv1.weight processed in: 0.013861894607543945 s
C shape: 128, 4
Layer layer2.1.conv2.weight processed in: 0.014148950576782227 s
C shape: 128, 4
Layer lay

In [ ]:
def matrix_to_partition(P, original_partition, previous_partition):
    """
    Translates an assignment matrix P (neurons by machines) back into a partition dictionary.

    Args:
        P (np.ndarray): A binary matrix of shape (num_neurons, num_partitions), where:
                        - P[i, j] = 1 if output neuron `i` is executed on machine `j`, 0 otherwise.
        original_partition (dict): The original partition dictionary, used to retrieve:
                                   - maps: Communication cost between partitions
        previous_partition (dict): The previous layer partition dictionary, used to retrieve:
                                   - channel_id: Input channels per partition

    Returns:
        dict: A new partition dictionary reconstructed based on P.
    """
    # Ensure P is a NumPy array
    P = np.array(P)

    # Validate input dimensions
    num_neurons, num_partitions = P.shape
    if 'num' not in original_partition or original_partition['num'] != num_partitions:
        raise ValueError("Mismatch between P's number of partitions and the original partition dictionary.")

    # Reconstruct the partition dictionary
    new_partition = {
        'num': num_partitions,
        'filter_id': [],  # Output neurons (filters) per partition
        'channel_id': previous_partition['filter_id'],  # Retain original input channels
        'maps': original_partition['maps'],  # Retain original communication cost map
    }

    # Populate filter_id for each partition
    for j in range(num_partitions):
        new_partition['filter_id'].append(np.where(P[:, j] == 1)[0])

    return new_partition


In [ ]:
P = np.array([
    [1, 0, 0],  # Neuron 0 -> Partition 0
    [1, 0, 0],  # Neuron 1 -> Partition 0
    [0, 1, 0],  # Neuron 2 -> Partition 1
    [0, 1, 0],  # Neuron 3 -> Partition 1
    [0, 0, 1],  # Neuron 4 -> Partition 2
    [0, 0, 1],  # Neuron 5 -> Partition 2
])

original_partition = {
    'num': 3,
    'filter_id': [np.array([0, 1]), np.array([2, 3]), np.array([4, 5])],
    'channel_id': [np.array([0, 1]), np.array([2]), np.array([3])],
    'maps': [[0, 1, 2], [1, 0, 1], [2, 1, 0]],
}

In [ ]:
new_partition = matrix_to_partition(P, original_partition)
print(new_partition)

In [2]:
class SingleConvModel(nn.Module):
    """
    A single Conv2D layer model that we can manipulate easily.
    """
    def __init__(self, in_channels=4, out_channels=4):
        super(SingleConvModel, self).__init__()
        # kernel_size=1 for simplicity, so weight shape = (out_channels, in_channels, 1, 1)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        return self.conv(x)


def create_suboptimal_configs():
    """
    Creates a dummy 'configs' dictionary with a deliberately suboptimal initial partition 
    for a single layer 'conv.weight'.
    """
    # We'll define 2 partitions: 0 and 1.
    # Suppose initially we put output neurons [0,1] in partition 0, and [2,3] in partition 1,
    # but the actual weights will strongly favor the opposite assignment.

    configs = {
        'partition': {
            'conv.weight': {
                'num': 2,
                # DELIBERATELY suboptimal: let's put [0,1] in partition 0, [2,3] in partition 1
                'filter_id': [np.array([0, 1]), np.array([2, 3])],
                # We'll also assume some initial channel partition—say, in channels: 
                # partition 0 -> [0], partition 1 -> [1,2,3].
                'channel_id': [np.array([0]), np.array([1, 2, 3])],
                # Communication cost maps (2x2). Large cost for crossing partitions -> 10
                'maps': [
                    [0,  10],
                    [10, 0 ]
                ]
            }
        },
        # Possibly other fields, like 'comm_costs'
        'comm_costs': {}
    }
    return configs


def test_suboptimal_assignment():
    """
    Demonstrates how update_assignments can fix a deliberately suboptimal initial partition.
    """
    # 1. Create a single-layer model with 4 in-channels, 4 out-channels.
    model = SingleConvModel(in_channels=4, out_channels=4).cpu()

    # 2. Override the model's weights so that crossing partitions is obviously costly.
    #    Let's fill them with zeros except for a few specific channels:
    #
    #    - Out channels 0,1 rely heavily on in-channel 1 (which is initially in partition 1),
    #      so they'd *prefer* to be in partition 1 to avoid crossing cost.
    #    - Out channels 2,3 rely heavily on in-channel 0 (which is initially in partition 0),
    #      so they'd *prefer* to be in partition 0.
    #
    #    This is the reverse of our initial partition assignment.
    with torch.no_grad():
        # shape: (out_channels=4, in_channels=4, 1, 1)
        w = model.conv.weight
        w.zero_()
        # Make out channels 0,1 non-zero in in-channel 1
        w[0, 1, 0, 0] = 1.0
        w[1, 1, 0, 0] = 1.0
        # Make out channels 2,3 non-zero in in-channel 0
        w[2, 0, 0, 0] = 1.0
        w[3, 0, 0, 0] = 1.0

    # 3. Create a config with a deliberately suboptimal initial assignment
    configs = create_suboptimal_configs()

    # 4. Print the initial partitioning
    print("=== Initial Partition ===")
    print("filter_id:", [f.tolist() for f in configs['partition']['conv.weight']['filter_id']])
    print("channel_id:", [c.tolist() for c in configs['partition']['conv.weight']['channel_id']])

    # 5. Call update_assignments
    update_assignments(model, configs)

    # 6. Inspect the updated partition
    print("\n=== Updated Partition ===")
    new_partition = configs['partition']['conv.weight']
    print("filter_id:", [f.tolist() for f in new_partition['filter_id']])
    print("channel_id:", [c.tolist() for c in new_partition['channel_id']])

    # 7. We expect out channels [0,1] to have moved to partition 1
    #    and out channels [2,3] to have moved to partition 0 (or something that lowers cost).
    #    The exact result depends on your assignment logic.

    # Optionally add an assertion or debugging:
    # e.g., Check if [0,1] ended in partition 1
    p0 = set(new_partition['filter_id'][0])
    p1 = set(new_partition['filter_id'][1])
    print("\nPartitions after update: p0 =", p0, ", p1 =", p1)
    # You might want to confirm that 0,1 ended up in p1 and 2,3 ended up in p0
    # but the actual final assignment depends on your Hungarian logic & cost matrix structure.


test_suboptimal_assignment()


=== Initial Partition ===
filter_id: [[0, 1], [2, 3]]
channel_id: [[0], [1, 2, 3]]
[(0, 1, 0.0), (1, 1, 0.0), (2, 0, 0.0), (3, 0, 0.0)]

=== Updated Partition ===
filter_id: [[2, 3], [0, 1]]
channel_id: [[0], [1, 2, 3]]

Partitions after update: p0 = {2, 3} , p1 = {0, 1}


In [1]:
import torch

# Suppose you have your BatchNorm2dPartition defined as above:
class BatchNorm2dPartition(torch.nn.Module):
    def __init__(self, planes, num_partition=1, momentum=0.1):
        super(BatchNorm2dPartition, self).__init__()
        self.num_partition = num_partition
        self.k, self.m = divmod(planes, num_partition)
        
        self.bn_list = torch.nn.ModuleList(
            torch.nn.BatchNorm2d(self.k + int(i < self.m), momentum=momentum)
            for i in range(num_partition)
        )
        
    def forward(self, x):
        # Each sub-BN operates on a slice of channels
        out_list = [
            bn(
                x[:,
                  i*self.k + min(i, self.m):(i+1)*self.k + min(i+1, self.m),
                  :, :]
            )
            for i, bn in enumerate(self.bn_list)
        ]
        out = torch.cat(out_list, dim=1)
        return out

def check_bn_partition_shapes():
    # Example: 10 output channels, partitioned into 3 sub-BNs
    planes = 10
    num_partition = 3
    bn_partition = BatchNorm2dPartition(planes, num_partition=num_partition)
    
    # Print out the shapes for each sub-BN's weight and bias
    for i, bn in enumerate(bn_partition.bn_list):
        print(f"Sub-BN {i}: weight shape = {bn.weight.shape}, bias shape = {bn.bias.shape}")

if __name__ == "__main__":
    check_bn_partition_shapes()


Sub-BN 0: weight shape = torch.Size([4]), bias shape = torch.Size([4])
Sub-BN 1: weight shape = torch.Size([3]), bias shape = torch.Size([3])
Sub-BN 2: weight shape = torch.Size([3]), bias shape = torch.Size([3])


## Benchmarking

In [2]:
# Files to load
config_path = './config/cifar10.yaml' 
partition_path = './config/resnet18-np4.yaml' 

In [3]:
# Define config and partition files
configs = load_yaml(config_path)
configs['partition_path'] = partition_path
model = get_model_from_code(configs)
configs = partition_generator(configs, model)
configs['comm_costs'] = set_communication_cost(model, configs['partition'],)
#input_var = get_input_from_code(configs)
#model = model.to(configs['device'])
#configs['partition'] = featuremap_summary(model, configs['partition'], input_var)
train_loader, test_loader = get_dataset_from_code(configs['data_code'], configs['batch_size'])

num_partition: {'conv1.weight': 4, 'inputs': 4, 'layer1.0.conv1.weight': 4, 'layer1.0.conv2.weight': 4, 'layer1.1.conv1.weight': 4, 'layer1.1.conv2.weight': 4, 'layer2.0.conv1.weight': 4, 'layer2.0.conv2.weight': 4, 'layer2.0.shortcut.0.weight': 4, 'layer2.1.conv1.weight': 4, 'layer2.1.conv2.weight': 4, 'layer3.0.conv1.weight': 4, 'layer3.0.conv2.weight': 4, 'layer3.0.shortcut.0.weight': 4, 'layer3.1.conv1.weight': 4, 'layer3.1.conv2.weight': 4, 'layer4.0.conv1.weight': 4, 'layer4.0.conv2.weight': 4, 'layer4.0.shortcut.0.weight': 4, 'layer4.1.conv1.weight': 4, 'layer4.1.conv2.weight': 4}
ratio_partition: {'conv1.weight': [1, 1, 1, 1], 'inputs': [1, 1, 1, 1], 'layer1.0.conv1.weight': [1, 1, 1, 1], 'layer1.0.conv2.weight': [1, 1, 1, 1], 'layer1.1.conv1.weight': [1, 1, 1, 1], 'layer1.1.conv2.weight': [1, 1, 1, 1], 'layer2.0.conv1.weight': [1, 1, 1, 1], 'layer2.0.conv2.weight': [1, 1, 1, 1], 'layer2.0.shortcut.0.weight': [1, 1, 1, 1], 'layer2.1.conv1.weight': [1, 1, 1, 1], 'layer2.1.conv2.

/home/sirera.m/.local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 3, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [4]:
filepath = os.path.join(os.getcwd(), 'assets', 'models', "cifar10-resnet18.pt")
model_name = filepath[:-3]
model_name = model_name + '-reassign' + str(configs['reassign']) + '2.pt'
state_dict = torch.load(model_name, map_location=configs['device'])
model = load_state_dict(model, state_dict['model_state_dict'] if 'model_state_dict' in state_dict 
                               else state_dict['state_dict'] if 'state_dict' in state_dict else state_dict,)

In [5]:
def set_communication_cost_debug(model, partition):
    comm_costs = {}
    device = next(model.parameters()).device
    total_time = 0

    for name, W in model.named_parameters():
        if name in partition:
            start_time = time.time()

            weight = W.cpu().detach().numpy()
            shape = weight.shape
            cost_mask = np.zeros(shape).reshape(shape[0], shape[1], -1)
            
            for i in range(partition[name]['num']):
                for j in range(partition[name]['num']):
                    if i == j:
                        continue
                    maps = partition[name]['maps'][i][j]
                    cost_mask[partition[name]['filter_id'][i][:, None], partition[name]['channel_id'][j]] = maps

            comm_costs[name] = torch.from_numpy(cost_mask.reshape(shape)).to(device)
            
            layer_time = time.time() - start_time
            total_time += layer_time
            print(f"Layer {name} cost computation time: {layer_time:.4f}s")

    print(f"Total communication cost time: {total_time:.4f}s")
    return comm_costs


In [6]:
def benchmark_assignment(model, configs, num_runs=10):
    """
    Benchmark the time taken by the assignment and communication cost functions.
    """
    start_time = time.time()
    # 1) Use the built FX graph.
    # 2) Identify all add nodes + their two conv parents -> store in add_pairs.
    partition_dict = configs['partition']
    gm = fx.symbolic_trace(model)

    # 1) node_map: node->"layer_name.weight" for quick lookup
    node_map = build_node_map(gm)
    named_mods = dict(model.named_modules())

    # 2) Identify add pairs
    add_node_pairs_map = build_add_pairs(gm, partition_dict, node_map)
    # This is: add_node -> (layerA, layerB, relationship)

    # 3) Build sets to guide the logic
    conv_in_add = set()
    parents_in_add = set()
    for add_node, (layerA, layerB, rel) in add_node_pairs_map.items():
        conv_in_add.add(layerA)
        conv_in_add.add(layerB)
        if rel == "A_is_parent":
            parents_in_add.add(layerA)
        elif rel == "B_is_parent":
            parents_in_add.add(layerB)
    model_graph = {
        'graph': gm,
        'node_map': node_map,
        'named_mods': named_mods,
        'addition_nodes': add_node_pairs_map,
        'addition_set': conv_in_add,
        'parents': parents_in_add,
    }
    graph_construction_time = time.time() - start_time
    
    times = []
    for _ in range(num_runs):
        # Time update_assignments
        start_time = time.time()
        update_assignments_with_timing(model, configs, model_graph)
        end_time = time.time()
        assignment_time = end_time - start_time

        # Time set_communication_cost_debug
        start_time = time.time()
        comm_costs = set_communication_cost_debug(model, configs['partition'])
        end_time = time.time()
        comm_time = end_time - start_time

        times.append((assignment_time, comm_time))

    times = np.array(times)
    mean_assignment_time = times[:, 0].mean()
    mean_comm_time = times[:, 1].mean()
    std_assignment_time = times[:, 0].std()
    std_comm_time = times[:, 1].std()
    print(f"Graph Construction - {graph_construction_time}")
    print(f"Update Assignments - Mean: {mean_assignment_time:.4f}s, Std Dev: {std_assignment_time:.4f}s")
    print(f"Set Communication Costs - Mean: {mean_comm_time:.4f}s, Std Dev: {std_comm_time:.4f}s")
    return times


In [7]:
benchmark_assignment(model, configs, 1)

Cost matrix shape: (64, 4)
Expanded matrix shape: (64, 64)
cap = 16
[Timing] computeassignment: 0.1008s
Cost matrix shape: (64, 4)
Expanded matrix shape: (64, 64)
cap = 16
[Timing] computeassignment: 0.0491s
Timing for unify_assignments_for_add (layer1.0.conv2.weight, conv1.weight):
  get_parent_partition_A: 0.0001s
  get_parent_partition_B: 0.0001s
  compute_combined_cost_matrix: 0.0213s
  compute_assignment: 0.0492s
  list_to_partition: 0.0000s
  update_partition_dict: 0.0000s
  Total: 0.0706s
Cost matrix shape: (64, 4)
Expanded matrix shape: (64, 64)
cap = 16
[Timing] computeassignment: 0.0036s
Cost matrix shape: (64, 4)
Expanded matrix shape: (64, 64)
cap = 16
[Timing] computeassignment: 0.0035s
Cost matrix shape: (128, 4)
Expanded matrix shape: (128, 128)
cap = 32
[Timing] computeassignment: 0.0137s
Cost matrix shape: (128, 4)
Expanded matrix shape: (128, 128)
cap = 32
[Timing] computeassignment: 0.9159s
Timing for unify_assignments_for_add (layer2.0.conv2.weight, layer2.0.shortcu

array([[98.92592978,  0.10222411]])

In [13]:
model = get_model_from_code(configs)
for param in model.parameters():
    param.data = torch.randn_like(param)

In [14]:
# Resnet18
benchmark_assignment(model, configs, 1)

Cost matrix shape: (64, 4)
Expanded matrix shape: (64, 64)
cap = 16
[Timing] computeassignment: 0.0092s
Cost matrix shape: (64, 4)
Expanded matrix shape: (64, 64)
cap = 16
[Timing] computeassignment: 0.0900s
Timing for unify_assignments_for_add (layer1.0.conv2.weight, conv1.weight):
  get_parent_partition_A: 0.0002s
  get_parent_partition_B: 0.0001s
  compute_combined_cost_matrix: 0.0462s
  compute_assignment: 0.0900s
  list_to_partition: 0.0001s
  update_partition_dict: 0.0000s
  Total: 0.1366s
Cost matrix shape: (64, 4)
Expanded matrix shape: (64, 64)
cap = 16
[Timing] computeassignment: 0.0044s
Cost matrix shape: (64, 4)
Expanded matrix shape: (64, 64)
cap = 16
[Timing] computeassignment: 0.0040s
Cost matrix shape: (128, 4)
Expanded matrix shape: (128, 128)
cap = 32
[Timing] computeassignment: 0.0144s
Cost matrix shape: (128, 4)
Expanded matrix shape: (128, 128)
cap = 32
[Timing] computeassignment: 0.0145s
Timing for unify_assignments_for_add (layer2.0.conv2.weight, layer2.0.shortcu

array([[2.6837399 , 0.05853343]])

In [11]:
# Resnet101
benchmark_assignment(model, configs)

Timing Results:
layer1.0.conv1.weight: {'cost_matrix': 0.019197940826416016, 'assignment': 0.006144285202026367, 'total': 0.028778076171875}
add_node_add: {'unify_assignments': 0.11316084861755371}
layer1.1.conv1.weight: {'cost_matrix': 0.011726617813110352, 'assignment': 0.003849029541015625, 'total': 0.01720118522644043}
layer1.1.conv2.weight: {'cost_matrix': 0.011386632919311523, 'assignment': 0.0036313533782958984, 'total': 0.016591787338256836}
layer1.2.conv1.weight: {'cost_matrix': 0.010919809341430664, 'assignment': 0.0036263465881347656, 'total': 0.016065597534179688}
layer1.2.conv2.weight: {'cost_matrix': 0.010851621627807617, 'assignment': 0.0035157203674316406, 'total': 0.01580190658569336}
layer2.0.conv1.weight: {'cost_matrix': 0.020479440689086914, 'assignment': 0.013268709182739258, 'total': 0.03518414497375488}
add_node_add_3: {'unify_assignments': 0.05683135986328125}
layer2.1.conv1.weight: {'cost_matrix': 0.021544933319091797, 'assignment': 0.013633966445922852, 'total

array([[7.64930892, 0.24480367],
       [7.77896309, 0.21619582],
       [7.69401455, 0.20796895],
       [7.69223022, 0.20685363],
       [7.71159601, 0.2138052 ],
       [7.80084062, 0.20518231],
       [7.49951315, 0.20659351],
       [7.67197847, 0.21001649],
       [7.75536323, 0.20756984],
       [7.73589587, 0.20801377]])

In [17]:
# ESCNet
benchmark_assignment(model, configs)

Timing Results:
layer1.0.conv1.weight: {'cost_matrix': 0.017202377319335938, 'assignment': 0.0030145645141601562, 'total': 0.02174663543701172}
add_node_add: {'unify_assignments': 0.04927945137023926}
layer2.0.conv1.weight: {'cost_matrix': 0.01088094711303711, 'assignment': 0.0019598007202148438, 'total': 0.013266324996948242}
layer2.0.conv2.weight: {'cost_matrix': 0.010368108749389648, 'assignment': 0.0018534660339355469, 'total': 0.012607097625732422}
conv2.weight: {'cost_matrix': 0.009595394134521484, 'assignment': 0.0017578601837158203, 'total': 0.01172018051147461}
Total update_assignments time: 0.1087s
Layer conv1.weight cost computation time: 0.0015s
Layer conv2.weight cost computation time: 0.0002s
Layer layer1.0.conv1.weight cost computation time: 0.0002s
Layer layer1.0.conv2.weight cost computation time: 0.0004s
Layer layer2.0.conv1.weight cost computation time: 0.0002s
Layer layer2.0.conv2.weight cost computation time: 0.0004s
Layer linear1.weight cost computation time: 0.04

array([[0.10900831, 0.04777813],
       [0.06193399, 0.03597474],
       [0.0476985 , 0.02286482],
       [0.04908419, 0.02245522],
       [0.04754162, 0.0224719 ],
       [0.04759216, 0.02283597],
       [0.04752755, 0.02229667],
       [0.04769993, 0.02226114],
       [0.04788375, 0.02276492],
       [0.04735541, 0.02246332]])

In [15]:
import numpy as np

# Function to generate varied random 512x512 cost matrices
def generate_random_cost_matrix(size=512, sparsity=0.3, value_range=(1, 100)):
    """
    Generates a random cost matrix of given size with specified sparsity and value range.
    
    Parameters:
        size (int): The size of the square matrix.
        sparsity (float): Probability of an entry being zero (0 to 1).
        value_range (tuple): Range of values for non-zero entries.

    Returns:
        np.ndarray: Generated cost matrix.
    """
    matrix = np.random.randint(value_range[0], value_range[1] + 1, size=(size, size))
    
    # Introduce zeros based on sparsity
    mask = np.random.rand(size, size) < sparsity
    matrix[mask] = 0
    
    return matrix

# Generate a few sample matrices with different characteristics
cost_matrices = {
    "low sparsity (10%)": generate_random_cost_matrix(sparsity=0.1),
    "moderate sparsity (30%)": generate_random_cost_matrix(sparsity=0.3),
    "high sparsity (70%)": generate_random_cost_matrix(sparsity=0.7),
    "low values (1-10)": generate_random_cost_matrix(value_range=(1, 10)),
    "high values (100-1000)": generate_random_cost_matrix(value_range=(100, 1000))
}

for key, matrix in cost_matrices.items():
    print(f"{key} Timings:")
    computeassignment_with_timing(matrix)


low sparsity (10%) Timings:
Cost matrix shape: (512, 512)
Expanded matrix shape: (512, 512)
cap = 1
[Timing] computeassignment: 1.3288s
Cost matrix stats: {'shape': (512, 512), 'num_elements': 262144, 'num_zeros': 25977, 'frac_zeros': 0.09909439086914062, 'num_near_zero': 25977, 'frac_near_zero': 0.09909439086914062, 'min_value': 0, 'max_value': 100, 'mean_value': 45.518680572509766, 'std_value': 31.270720091369704}
moderate sparsity (30%) Timings:
Cost matrix shape: (512, 512)
Expanded matrix shape: (512, 512)
cap = 1
[Timing] computeassignment: 0.3997s
Cost matrix stats: {'shape': (512, 512), 'num_elements': 262144, 'num_zeros': 78915, 'frac_zeros': 0.3010368347167969, 'num_near_zero': 78915, 'frac_near_zero': 0.3010368347167969, 'min_value': 0, 'max_value': 100, 'mean_value': 35.269290924072266, 'std_value': 33.44876182475809}
high sparsity (70%) Timings:
Cost matrix shape: (512, 512)
Expanded matrix shape: (512, 512)
cap = 1
[Timing] computeassignment: 0.2118s
Cost matrix stats: {'